## 1. 라이브러리 로드

In [109]:
# 라이브러리 호출
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import platform
import ast
from collections import Counter
import json

warnings.filterwarnings('ignore')

# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # Mac
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 컬럼 너비 제한 해제
pd.set_option('display.max_colwidth', None)

---
## 2. 데이터 로드

In [110]:
path = "../../../tft_game_dataset/TFT_Platinum_MatchData.csv"
path2 = "../../../tft_game_dataset/TFT_Diamond_MatchData.csv"
path3 = "../../../tft_game_dataset/TFT_Master_MatchData.csv"
path4 = "../../../tft_game_dataset/TFT_GrandMaster_MatchData.csv"
path5 = "../../../tft_game_dataset/TFT_Challenger_MatchData.csv"
path6 = "../../../tft_game_dataset/TFT_Champion_CurrentVersion.csv"
path7 = "../../../tft_game_dataset/TFT_Item_CurrentVersion.csv"

df_platinum_Match = pd.read_csv(path)
df_diamond_Match = pd.read_csv(path2)
df_master_Match = pd.read_csv(path3)
df_grand_master_Match = pd.read_csv(path4)
df_challenger_Match = pd.read_csv(path5)

df_champion_info = pd.read_csv(path6)
df_items_info = pd.read_csv(path7)


In [111]:
# 매치관련 데이터가 담긴 딕셔너리 정의
match_data = {
    'platinum': df_platinum_Match,
    'diamond': df_diamond_Match,
    'master': df_master_Match,
    'grand_master': df_grand_master_Match,
    'challenger': df_challenger_Match,
}

# 각 티어별 테이블에 'Tier'정보가 담긴 컬럼 추가
for name, df in match_data.items():
    df['tier'] = name


# 모든 티어의 경기데이터가 담긴 데이터프레임 제작
# ignore_index=True: 데이터를 병합한 뒤, 새로운 인덱스 부여
df_all_match = pd.concat(match_data.values(), ignore_index=True)

df_all_match.head(3)

,gameId,gameDuration,level,lastRound,Ranked,ingameDuration,combination,champion,tier
0,KR_4291707834,1963.905273,6,27,5,1390.165771,"{'Cybernetic': 1, 'Demolitionist': 1, 'Infiltrator': 1, 'Rebel': 1, 'Set3_Brawler': 1, 'Set3_Celestial': 1, 'Set3_Void': 1, 'Sniper': 1}","{'Ziggs': {'items': [7], 'star': 1}, 'Ashe': {'items': [9], 'star': 1}, 'ChoGath': {'items': [6], 'star': 1}, 'Ekko': {'items': [1], 'star': 1}}",platinum
1,KR_4291707834,1963.905273,8,37,3,1891.282715,"{'Blaster': 1, 'Chrono': 1, 'Cybernetic': 4, 'Demolitionist': 1, 'Rebel': 1, 'Set3_Blademaster': 2, 'Set3_Brawler': 1, 'Set3_Sorcerer': 1, 'Set3_Void': 1, 'Valkyrie': 1, 'Vanguard': 2}","{'Ziggs': {'items': [24], 'star': 3}, 'Fiora': {'items': [37], 'star': 2}, 'Leona': {'items': [36, 24], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [5], 'star': 2}, 'Kayle': {'items': [], 'star': 2}, 'WuKong': {'items': [3, 67], 'star': 2}, 'VelKoz': {'items': [4], 'star': 2}}",platinum
2,KR_4291707834,1963.905273,6,25,7,1279.461060,"{'Blaster': 1, 'Cybernetic': 1, 'DarkStar': 2, 'Infiltrator': 1, 'Mercenary': 1, 'Set3_Blademaster': 1, 'Set3_Mystic': 1, 'Valkyrie': 1}","{'Fiora': {'items': [1], 'star': 1}, 'Shaco': {'items': [6], 'star': 1}, 'Karma': {'items': [4], 'star': 1}, 'MissFortune': {'items': [3], 'star': 1}}",platinum


---
## 3. 데이터 전처리
### 3-1. 중복행 제거

In [112]:
# 중복행 제거
duplicates = df_all_match[df_all_match.duplicated(keep=False)]

print(f"중복 제거 전: {len(df_all_match)}")
print(f"중복된 행 개수: {len(duplicates)}")

# 결과 확인
df_all_match = df_all_match.drop_duplicates()
print(f"\n중복 제거 후: {len(df_all_match)}")

중복 제거 전: 399998
중복된 행 개수: 80

중복 제거 후: 399930


### 3-2. 컬럼명 소문자로 통일

In [113]:
# 전체 컬럼명 소문자 통일
df_all_match.columns = df_all_match.columns.str.lower()

print(df_all_match.columns)

Index(['gameid', 'gameduration', 'level', 'lastround', 'ranked',
       'ingameduration', 'combination', 'champion', 'tier'],
      dtype='str')


### 3-3. ranked=0인 데이터 삭제

In [114]:
# ranked = 0이 포함된 경기 데이터 삭제
df_all_match_2 = df_all_match.copy()

# ranked==0인 행의 gameId 추출
drop_game_ids = df_all_match_2[df_all_match_2['ranked']==0]['gameid'].unique()

# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx = df_all_match_2[df_all_match_2['gameid'].isin(drop_game_ids)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx)

print(f"ranked = 0인 데이터가 포함된 경기 데이터 삭제하기 전: {df_all_match.shape}")
print(f"ranked = 0인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

ranked = 0인 데이터가 포함된 경기 데이터 삭제하기 전: (399930, 9)
ranked = 0인 데이터가 포함된 경기 데이터 삭제한 후: (399886, 9)


### 3-4. 경기시간 = 0인 데이터 삭제

In [115]:
# 전체 게임시간 = 0인 행의 GameId 추출
zero_duration_ids = set(df_all_match_2[df_all_match_2['gameduration'] == 0]['gameid'])

# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx_2 = df_all_match_2[df_all_match_2['gameid'].isin(zero_duration_ids)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx_2)

print(f"gameduration = 0인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

gameduration = 0인 데이터가 포함된 경기 데이터 삭제한 후: (399886, 9)


### 3-5. 경기 참여 유저 수가 8명 미만인 게임 삭제

In [116]:
# 게임 ID별 플레이어 수 정보 추가
df_all_match_2['player_cnt'] = df_all_match_2['gameid'].map(df_all_match_2['gameid'].value_counts())

# player_cnt < 8인 행의 gameId 추출
drop_game_ids_3 = df_all_match_2[df_all_match_2['player_cnt'] < 8]['gameid'].unique()

# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx_3 = df_all_match_2[df_all_match_2['gameid'].isin(drop_game_ids_3)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx_3)

print(f"player_cnt < 8인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

player_cnt < 8인 데이터가 포함된 경기 데이터 삭제한 후: (399872, 10)


### 3-6. 시즌2 데이터 삭제

In [117]:
# 시즌 2 고유 시너지 목록
season2_keys = {'Alchemist', 'Avatar', 'Berserker',
                'Crystal', 'Desert', 'Druid', 'Electric',
                'Light', 'Mage', 'Mountain', 'Mystic', 'Ocean',
                'Poison', 'Predator', 'Set2_Assassin', 'Set2_Blademaster',
                'Set2_Glacial', 'Set2_Ranger', 'Shadow', 'Soulbound',
                'Summoner', 'Warden', 'Wind', 'Woodland'}  # 시즌 2 시너지 입력


# 시즌 2 시너지가 하나라도 포함되면 시즌 2로 분류
df_all_match_2['season'] = df_all_match_2['combination'].apply(
    lambda x: 'season 2' if any(k in season2_keys for k in ast.literal_eval(x).keys())
    else 'season 3'
)

# season 2인 행의 gameId 추출
drop_game_ids_4 = df_all_match_2[df_all_match_2['season'] == 'season 2']['gameid'].unique()


# 해당 gameId를 가진 행의 인덱스 추출 후 삭제
drop_idx_4 = df_all_match_2[df_all_match_2['gameid'].isin(drop_game_ids_4)].index
df_all_match_2 = df_all_match_2.drop(index=drop_idx_4)

print(f"season 2인 데이터가 포함된 경기 데이터 삭제한 후: {df_all_match_2.shape}")

season 2인 데이터가 포함된 경기 데이터 삭제한 후: (396640, 11)


### 3-7. 시즌 3 시너지 더미 데이터 ”TemplateTrait” 삭제
- 시너지 데이터 중 해당 값인 `{'TemplateTrait': 1}`만 삭제

In [118]:
# TemplateTrait 키가 있는 행만 필터링 후 값 확인
df_all_match_2[df_all_match_2['combination'].apply(
    lambda x: 'TemplateTrait' in ast.literal_eval(x).keys() # TemplateTrait 키가 있는 행만 필터링
)]['combination'].apply(                                    # 필터링된 행 중에서 combination 컬럼만 가져옴
    lambda x: ast.literal_eval(x).get('TemplateTrait')      # TemplateTrait 키에 할당된 값을 가져옴
).value_counts()


combination
1    65987
Name: count, dtype: int64

In [119]:
# 분석 목적에 영향을 주지 않는 dummy 데이터로 판단하여 시너지 중 'TemplateTrait'만 삭제
df_all_match_2['combination'] = df_all_match_2['combination'].apply(
    lambda x: json.dumps(
        # key: value = 키-값의 쌍을 의미함
        {key: value for key, value in ast.literal_eval(x).items() if key != 'TemplateTrait'},
        ensure_ascii=False
    )
)

In [120]:
# TemplateTrait 키가 있는 행 수 확인 → (0,11)이면 완전히 삭제된 것
df_all_match_2[df_all_match_2['combination'].apply(
    lambda x: 'TemplateTrait' in ast.literal_eval(x).keys()
)].shape

(0, 11)

### 3-8. 유저 ID 컬럼 제작

In [121]:
df_all_match_2['user_id'] = 'KR-USER-' + (df_all_match_2.index + 1).astype(str)

In [122]:
df_all_match_2['user_id']

0              KR-USER-1
1              KR-USER-2
2              KR-USER-3
3              KR-USER-4
4              KR-USER-5
               ...      
399993    KR-USER-399994
399994    KR-USER-399995
399995    KR-USER-399996
399996    KR-USER-399997
399997    KR-USER-399998
Name: user_id, Length: 396640, dtype: str

### 3-9. 티어가 섞인 경기 정보 추출

In [123]:
# 티어가 섞인 gameId 추출
mixed_gameids = df_all_match_2.groupby('gameid')['tier'].nunique() # gameid별로 tier의 고유값 개수를 계산
mixed_gameids = mixed_gameids[mixed_gameids > 1].index # 고유값이 2 이상인 gameId만 추출

# 해당 gameId의 티어 구성 확인
# mixed_gameids에 포함된 gameId를 가진 행만 추출 -> gameid별로 어떤 티어가 몇 명씩 있는지 카운트
df_all_match_2[df_all_match_2['gameid'].isin(mixed_gameids)].groupby('gameid')['tier'].value_counts()

gameid         tier        
KR_4263820118  platinum        8
               master          8
KR_4313697214  platinum        8
               master          8
KR_4320079059  platinum        8
               diamond         8
KR_4344513862  diamond         8
               master          8
KR_4347884427  diamond         8
               master          8
KR_4357966612  platinum        8
               grand_master    8
KR_4358922415  diamond         8
               master          8
KR_4361594426  diamond         8
               master          8
KR_4361773461  diamond         8
               grand_master    8
KR_4362846604  diamond         8
               master          8
KR_4364453473  diamond         8
               grand_master    8
KR_4365284161  diamond         8
               master          8
KR_4378896137  platinum        8
               diamond         8
KR_4381231118  platinum        8
               diamond         8
KR_4387025966  platinum        8
               

In [124]:
# 티어 순서 정의 (숫자가 높을수록 높은 티어)
tier_order = {
    'platinum': 1,
    'diamond': 2,
    'master': 3,
    'grand_master': 4,
    'challenger': 5
}

### 여기서부턴 제겁니다 껄껄(이젠 아닙니다 ㅠㅠ)

In [125]:
df_all_match_2.head()

,gameid,gameduration,level,lastround,ranked,ingameduration,combination,champion,tier,player_cnt,season,user_id
0,KR_4291707834,1963.905273,6,27,5,1390.165771,"{""Cybernetic"": 1, ""Demolitionist"": 1, ""Infiltrator"": 1, ""Rebel"": 1, ""Set3_Brawler"": 1, ""Set3_Celestial"": 1, ""Set3_Void"": 1, ""Sniper"": 1}","{'Ziggs': {'items': [7], 'star': 1}, 'Ashe': {'items': [9], 'star': 1}, 'ChoGath': {'items': [6], 'star': 1}, 'Ekko': {'items': [1], 'star': 1}}",platinum,8,season 3,KR-USER-1
1,KR_4291707834,1963.905273,8,37,3,1891.282715,"{""Blaster"": 1, ""Chrono"": 1, ""Cybernetic"": 4, ""Demolitionist"": 1, ""Rebel"": 1, ""Set3_Blademaster"": 2, ""Set3_Brawler"": 1, ""Set3_Sorcerer"": 1, ""Set3_Void"": 1, ""Valkyrie"": 1, ""Vanguard"": 2}","{'Ziggs': {'items': [24], 'star': 3}, 'Fiora': {'items': [37], 'star': 2}, 'Leona': {'items': [36, 24], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [5], 'star': 2}, 'Kayle': {'items': [], 'star': 2}, 'WuKong': {'items': [3, 67], 'star': 2}, 'VelKoz': {'items': [4], 'star': 2}}",platinum,8,season 3,KR-USER-2
2,KR_4291707834,1963.905273,6,25,7,1279.461060,"{""Blaster"": 1, ""Cybernetic"": 1, ""DarkStar"": 2, ""Infiltrator"": 1, ""Mercenary"": 1, ""Set3_Blademaster"": 1, ""Set3_Mystic"": 1, ""Valkyrie"": 1}","{'Fiora': {'items': [1], 'star': 1}, 'Shaco': {'items': [6], 'star': 1}, 'Karma': {'items': [4], 'star': 1}, 'MissFortune': {'items': [3], 'star': 1}}",platinum,8,season 3,KR-USER-3
3,KR_4291707834,1963.905273,7,38,2,1955.608521,"{""DarkStar"": 1, ""Protector"": 2, ""Set3_Blademaster"": 1, ""Set3_Celestial"": 5, ""Set3_Mystic"": 1, ""Sniper"": 1, ""StarGuardian"": 2, ""Vanguard"": 2}","{'Poppy': {'items': [], 'star': 2}, 'Xayah': {'items': [19, 23, 19], 'star': 3}, 'Rakan': {'items': [], 'star': 2}, 'XinZhao': {'items': [16], 'star': 2}, 'Mordekaiser': {'items': [35, 67, 33], 'star': 3}, 'Ashe': {'items': [], 'star': 2}, 'Soraka': {'items': [68, 47], 'star': 2}}",platinum,8,season 3,KR-USER-4
4,KR_4291707834,1963.905273,8,38,1,1955.608521,"{""Blaster"": 1, ""Chrono"": 5, ""DarkStar"": 3, ""Protector"": 1, ""Set3_Blademaster"": 1, ""Set3_Brawler"": 1, ""Set3_Sorcerer"": 2, ""Sniper"": 2}","{'TwistedFate': {'items': [36, 27], 'star': 3}, 'Caitlyn': {'items': [49, 29], 'star': 2}, 'JarvanIV': {'items': [56], 'star': 2}, 'Blitzcrank': {'items': [15], 'star': 2}, 'Shen': {'items': [77, 6], 'star': 2}, 'Ezreal': {'items': [16], 'star': 2}, 'Lux': {'items': [], 'star': 2}, 'Jhin': {'items': [], 'star': 2}}",platinum,8,season 3,KR-USER-5


In [126]:
df_all_match_2['combination'].dtype

<StringDtype(na_value=nan)>

In [127]:
df_all_match_2['champion'].dtype

<StringDtype(na_value=nan)>

In [128]:
#str -> dic로 변경
import ast

df_all_match_2['combination'] = df_all_match_2['combination'].apply(ast.literal_eval)
df_all_match_2['champion'] = df_all_match_2['champion'].apply(ast.literal_eval)

In [129]:
print(type(df_all_match_2.loc[0, 'combination']))
print(type(df_all_match_2.loc[0, 'champion']))

<class 'dict'>
<class 'dict'>


In [130]:
mask = (
    (df_all_match_2['champion'] != {}) &
    (df_all_match_2['combination'] == {})
)

df_all_match_2[mask].head()

,gameid,gameduration,level,lastround,ranked,ingameduration,combination,champion,tier,player_cnt,season,user_id
6627,KR_4271274609,2037.501343,6,13,8,637.688232,{},"{'Fiora': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 2}, 'KaiSa': {'items': [], 'star': 1}, 'Mordekaiser': {'items': [], 'star': 2}, 'Kassadin': {'items': [], 'star': 1}}",platinum,8,season 3,KR-USER-6628
10005,KR_4359182660,2064.300293,5,13,8,621.877319,{},"{'JarvanIV': {'items': [], 'star': 1}, 'Lux': {'items': [], 'star': 1}}",platinum,8,season 3,KR-USER-10006
15897,KR_4320179268,2129.208252,4,12,8,604.033447,{},"{'Malphite': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 1}, 'Mordekaiser': {'items': [], 'star': 1}, 'Ashe': {'items': [], 'star': 1}}",platinum,8,season 3,KR-USER-15898
20909,KR_4380989360,2041.039062,8,37,2,2032.800903,{},"{'Fiora': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [], 'star': 2}, 'Irelia': {'items': [16, 19, 29], 'star': 2}, 'WuKong': {'items': [], 'star': 2}, 'Thresh': {'items': [], 'star': 1}, 'Ekko': {'items': [], 'star': 2}}",platinum,8,season 3,KR-USER-20910
22014,KR_4366152411,1980.549683,7,24,8,1259.300781,{},"{'Malphite': {'items': [], 'star': 2}, 'KhaZix': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 3}, 'Mordekaiser': {'items': [], 'star': 2}, 'Shaco': {'items': [], 'star': 2}, 'Jayce': {'items': [], 'star': 1}, 'WuKong': {'items': [], 'star': 2}}",platinum,8,season 3,KR-USER-22015


In [131]:
mask.sum()

np.int64(117)

In [132]:
df_all_match_2.loc[mask, ['gameid', 'champion', 'combination', 'tier', 'season']].head(10)

,gameid,champion,combination,tier,season
6627,KR_4271274609,"{'Fiora': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 2}, 'KaiSa': {'items': [], 'star': 1}, 'Mordekaiser': {'items': [], 'star': 2}, 'Kassadin': {'items': [], 'star': 1}}",{},platinum,season 3
10005,KR_4359182660,"{'JarvanIV': {'items': [], 'star': 1}, 'Lux': {'items': [], 'star': 1}}",{},platinum,season 3
15897,KR_4320179268,"{'Malphite': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 1}, 'Mordekaiser': {'items': [], 'star': 1}, 'Ashe': {'items': [], 'star': 1}}",{},platinum,season 3
20909,KR_4380989360,"{'Fiora': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [], 'star': 2}, 'Irelia': {'items': [16, 19, 29], 'star': 2}, 'WuKong': {'items': [], 'star': 2}, 'Thresh': {'items': [], 'star': 1}, 'Ekko': {'items': [], 'star': 2}}",{},platinum,season 3
22014,KR_4366152411,"{'Malphite': {'items': [], 'star': 2}, 'KhaZix': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 3}, 'Mordekaiser': {'items': [], 'star': 2}, 'Shaco': {'items': [], 'star': 2}, 'Jayce': {'items': [], 'star': 1}, 'WuKong': {'items': [], 'star': 2}}",{},platinum,season 3
41248,KR_4367362223,"{'Darius': {'items': [], 'star': 2}, 'Rakan': {'items': [], 'star': 2}, 'XinZhao': {'items': [], 'star': 2}, 'Jayce': {'items': [], 'star': 3}, 'Kassadin': {'items': [], 'star': 2}, 'Ashe': {'items': [], 'star': 2}, 'Jhin': {'items': [], 'star': 1}}",{},platinum,season 3
42176,KR_4289735897,"{'Malphite': {'items': [], 'star': 2}, 'Fiora': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 2}, 'Yasuo': {'items': [], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [], 'star': 2}, 'Irelia': {'items': [], 'star': 1}, 'AurelionSol': {'items': [], 'star': 2}}",{},platinum,season 3
45966,KR_4345979831,"{'Malphite': {'items': [], 'star': 2}, 'KhaZix': {'items': [], 'star': 2}, 'Graves': {'items': [], 'star': 2}, 'KaiSa': {'items': [], 'star': 2}, 'Shaco': {'items': [], 'star': 2}}",{},platinum,season 3
71470,KR_4294692860,"{'TwistedFate': {'items': [], 'star': 2}, 'Malphite': {'items': [], 'star': 2}, 'KhaZix': {'items': [], 'star': 2}, 'Blitzcrank': {'items': [], 'star': 2}, 'Vi': {'items': [], 'star': 1}, 'ChoGath': {'items': [], 'star': 2}, 'VelKoz': {'items': [], 'star': 2}}",{},platinum,season 3
85620,KR_4400073539,"{'KhaZix': {'items': [], 'star': 2}, 'KaiSa': {'items': [], 'star': 2}, 'Annie': {'items': [], 'star': 2}, 'Shaco': {'items': [], 'star': 2}, 'Rumble': {'items': [], 'star': 1}, 'WuKong': {'items': [], 'star': 2}}",{},diamond,season 3


In [133]:
#전체 시너지 조회
all_traits = set()

for x in df_all_match_2['combination']:
    all_traits.update(x.keys())

all_traits

{'Blaster',
 'Chrono',
 'Cybernetic',
 'DarkStar',
 'Demolitionist',
 'Infiltrator',
 'ManaReaver',
 'MechPilot',
 'Mercenary',
 'Protector',
 'Rebel',
 'Set3_Blademaster',
 'Set3_Brawler',
 'Set3_Celestial',
 'Set3_Mystic',
 'Set3_Sorcerer',
 'Set3_Void',
 'Sniper',
 'SpacePirate',
 'StarGuardian',
 'Starship',
 'Valkyrie',
 'Vanguard'}

In [134]:
#전체 챔피언 조회
all_champion = set()

for x in df_all_match_2['champion']:
    all_champion.update(x.keys())

all_champion

{'Ahri',
 'Annie',
 'Ashe',
 'AurelionSol',
 'Blitzcrank',
 'Caitlyn',
 'ChoGath',
 'Darius',
 'Ekko',
 'Ezreal',
 'Fiora',
 'Fizz',
 'Gangplank',
 'Graves',
 'Irelia',
 'JarvanIV',
 'Jayce',
 'Jhin',
 'Jinx',
 'KaiSa',
 'Karma',
 'Kassadin',
 'Kayle',
 'KhaZix',
 'Leona',
 'Lucian',
 'Lulu',
 'Lux',
 'Malphite',
 'MasterYi',
 'MissFortune',
 'Mordekaiser',
 'Neeko',
 'Poppy',
 'Rakan',
 'Rumble',
 'Shaco',
 'Shen',
 'Sona',
 'Soraka',
 'Syndra',
 'Thresh',
 'TwistedFate',
 'VelKoz',
 'Vi',
 'WuKong',
 'Xayah',
 'Xerath',
 'XinZhao',
 'Yasuo',
 'Ziggs',
 'Zoe'}

In [135]:
champion_traits = {
    'Ahri': ['StarGuardian', 'Set3_Sorcerer'],
    'Annie': ['MechPilot', 'Set3_Sorcerer'],
    'Ashe': ['Set3_Celestial', 'Sniper'],
    'AurelionSol': ['Rebel', 'Starship'],
    'Blitzcrank': ['Chrono', 'Set3_Brawler'],
    'Caitlyn': ['Chrono', 'Sniper'],
    'ChoGath': ['Set3_Void', 'Set3_Brawler'],
    'Darius': ['SpacePirate', 'ManaReaver'],
    'Ekko': ['Cybernetic', 'Infiltrator'],
    'Ezreal': ['Chrono', 'Blaster'],
    'Fiora': ['Cybernetic', 'Set3_Blademaster'],
    'Fizz': ['MechPilot', 'Infiltrator'],
    'Gangplank': ['SpacePirate', 'Demolitionist', 'Mercenary'],
    'Graves': ['SpacePirate', 'Blaster'],
    'Irelia': ['Cybernetic', 'Set3_Blademaster', 'ManaReaver'],
    'JarvanIV': ['DarkStar', 'Protector'],
    'Jayce': ['SpacePirate', 'Vanguard'],
    'Jhin': ['DarkStar', 'Sniper'],
    'Jinx': ['Rebel', 'Blaster'],
    'KaiSa': ['Valkyrie', 'Infiltrator'],
    'Karma': ['DarkStar', 'Set3_Mystic'],
    'Kassadin': ['Set3_Celestial', 'ManaReaver'],
    'Kayle': ['Valkyrie', 'Set3_Blademaster'],
    'KhaZix': ['Set3_Void', 'Infiltrator'],
    'Leona': ['Cybernetic', 'Vanguard'],
    'Lucian': ['Cybernetic', 'Blaster'],
    'Lulu': ['Set3_Celestial', 'Set3_Mystic'],
    'Lux': ['DarkStar', 'Set3_Sorcerer'],
    'Malphite': ['Rebel', 'Set3_Brawler'],
    'MasterYi': ['Rebel', 'Set3_Blademaster'],
    'MissFortune': ['Valkyrie', 'Blaster', 'Mercenary'],
    'Mordekaiser': ['DarkStar', 'Vanguard'],
    'Neeko': ['StarGuardian', 'Protector'],
    'Poppy': ['StarGuardian', 'Vanguard'],
    'Rakan': ['Set3_Celestial', 'Protector'],
    'Rumble': ['MechPilot', 'Demolitionist'],
    'Shaco': ['DarkStar', 'Infiltrator'],
    'Shen': ['Chrono', 'Set3_Blademaster'],
    'Sona': ['Rebel', 'Set3_Mystic'],
    'Soraka': ['StarGuardian', 'Set3_Mystic'],
    'Syndra': ['StarGuardian', 'Set3_Sorcerer'],
    'Thresh': ['Chrono', 'ManaReaver'],
    'TwistedFate': ['Chrono', 'Set3_Sorcerer'],
    'VelKoz': ['Set3_Void', 'Set3_Sorcerer'],
    'Vi': ['Cybernetic', 'Set3_Brawler'],
    'WuKong': ['Chrono', 'Vanguard'],
    'Xayah': ['Set3_Celestial', 'Set3_Blademaster'],
    'Xerath': ['DarkStar', 'Set3_Sorcerer'],
    'XinZhao': ['Set3_Celestial', 'Protector'],
    'Yasuo': ['Rebel', 'Set3_Blademaster'],
    'Ziggs': ['Rebel', 'Demolitionist'],
    'Zoe': ['StarGuardian', 'Set3_Sorcerer']
}


In [136]:
#뒤집개템
item_to_trait = {
    18: 'Set3_Blademaster',   # Blade of the Ruined King
    28: 'Infiltrator',        # Infiltrator's Talons
    38: 'Demolitionist',      # Demolitionist's Charge
    48: 'StarGuardian',       # Star Guardian's Charm
    58: 'Rebel',              # Rebel Medal
    68: 'Set3_Celestial',     # Celestial Orb
    78: 'Protector',          # Protector's Chestguard
    89: 'DarkStar'            # Dark Star's Heart
}

In [137]:
# champion 컬럼의 챔피언 목록을 이용해
# 각 시너지(trait)가 몇 번 등장하는지 세어 combination 형태로 만드는 함수v1

def make_combination_from_champion(champion_dict):
    #시너지별 개수를 저장할 빈 딕셔너리 생성
    trait_count = {}
    
    #챔피언 이름 하나씩 반복
    for champ_name in champion_dict.keys():

        # 챔피언별 기본 시너지 조회
        traits = champion_traits.get(champ_name, [])
        
        # 해당 챔피언의 시너지를 하나씩 확인
        for trait in traits:
            # trait_count에 이미 있으면 기존 값 +1
            # 없으면 0부터 시작해서 +1
            trait_count[trait] = trait_count.get(trait, 0) + 1
            
     # 시너지별 개수 반환
    return trait_count

In [138]:
# 원본 컬럼유지, 새로운 컬럼 추가
df_all_match_2['combination_rebuilt'] = df_all_match_2['combination'].copy()

In [139]:
# champion이 O & combination이 X인 갯수
mask = (
    (df_all_match_2['champion'] != {}) &
    (df_all_match_2['combination'] == {})
)

print("채우기 대상 개수:", mask.sum())

채우기 대상 개수: 117


In [140]:
#함수v1 실행
df_all_match_2.loc[mask, 'combination_rebuilt'] = (
    df_all_match_2.loc[mask, 'champion']
    .apply(make_combination_from_champion)
)

# 개수 계산
original_empty = (df_all_match_2['combination'] == {}).sum()
rebuilt_empty = (df_all_match_2['combination_rebuilt'] == {}).sum()
filled_count = original_empty - rebuilt_empty
total_rows = len(df_all_match_2)

# 비율 계산
original_empty_ratio = original_empty / total_rows * 100
rebuilt_empty_ratio = rebuilt_empty / total_rows * 100
filled_ratio_total = filled_count / total_rows * 100
filled_ratio_from_empty = filled_count / original_empty * 100 if original_empty != 0 else 0

# 출력
print("=========함수v1 실행 결과=========")
print(f"원본 combination 빈값 개수: {original_empty} ({original_empty_ratio:.2f}%)")
print(f"새 컬럼 combination_rebuilt 빈값 개수: {rebuilt_empty} ({rebuilt_empty_ratio:.2f}%)")
print(f"채운 개수: {filled_count} ({filled_ratio_total:.2f}% / 빈값 기준 {filled_ratio_from_empty:.2f}%)")

=========함수v1 실행 결과=========
원본 combination 빈값 개수: 380 (0.10%)
새 컬럼 combination_rebuilt 빈값 개수: 263 (0.07%)
채운 개수: 117 (0.03% / 빈값 기준 30.79%)


In [141]:
df_all_match_2.loc[mask, ['gameid', 'champion', 'combination', 'combination_rebuilt']].head(10)

,gameid,champion,combination,combination_rebuilt
6627,KR_4271274609,"{'Fiora': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 2}, 'KaiSa': {'items': [], 'star': 1}, 'Mordekaiser': {'items': [], 'star': 2}, 'Kassadin': {'items': [], 'star': 1}}",{},"{'Cybernetic': 2, 'Set3_Blademaster': 1, 'Vanguard': 2, 'Valkyrie': 1, 'Infiltrator': 1, 'DarkStar': 1, 'Set3_Celestial': 1, 'ManaReaver': 1}"
10005,KR_4359182660,"{'JarvanIV': {'items': [], 'star': 1}, 'Lux': {'items': [], 'star': 1}}",{},"{'DarkStar': 2, 'Protector': 1, 'Set3_Sorcerer': 1}"
15897,KR_4320179268,"{'Malphite': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 1}, 'Mordekaiser': {'items': [], 'star': 1}, 'Ashe': {'items': [], 'star': 1}}",{},"{'Rebel': 1, 'Set3_Brawler': 1, 'Cybernetic': 1, 'Vanguard': 2, 'DarkStar': 1, 'Set3_Celestial': 1, 'Sniper': 1}"
20909,KR_4380989360,"{'Fiora': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [], 'star': 2}, 'Irelia': {'items': [16, 19, 29], 'star': 2}, 'WuKong': {'items': [], 'star': 2}, 'Thresh': {'items': [], 'star': 1}, 'Ekko': {'items': [], 'star': 2}}",{},"{'Cybernetic': 6, 'Set3_Blademaster': 2, 'Vanguard': 2, 'Blaster': 1, 'Set3_Brawler': 1, 'ManaReaver': 2, 'Chrono': 2, 'Infiltrator': 1}"
22014,KR_4366152411,"{'Malphite': {'items': [], 'star': 2}, 'KhaZix': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 3}, 'Mordekaiser': {'items': [], 'star': 2}, 'Shaco': {'items': [], 'star': 2}, 'Jayce': {'items': [], 'star': 1}, 'WuKong': {'items': [], 'star': 2}}",{},"{'Rebel': 1, 'Set3_Brawler': 1, 'Set3_Void': 1, 'Infiltrator': 2, 'Cybernetic': 1, 'Vanguard': 4, 'DarkStar': 2, 'SpacePirate': 1, 'Chrono': 1}"
41248,KR_4367362223,"{'Darius': {'items': [], 'star': 2}, 'Rakan': {'items': [], 'star': 2}, 'XinZhao': {'items': [], 'star': 2}, 'Jayce': {'items': [], 'star': 3}, 'Kassadin': {'items': [], 'star': 2}, 'Ashe': {'items': [], 'star': 2}, 'Jhin': {'items': [], 'star': 1}}",{},"{'SpacePirate': 2, 'ManaReaver': 2, 'Set3_Celestial': 4, 'Protector': 2, 'Vanguard': 1, 'Sniper': 2, 'DarkStar': 1}"
42176,KR_4289735897,"{'Malphite': {'items': [], 'star': 2}, 'Fiora': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 2}, 'Yasuo': {'items': [], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [], 'star': 2}, 'Irelia': {'items': [], 'star': 1}, 'AurelionSol': {'items': [], 'star': 2}}",{},"{'Rebel': 3, 'Set3_Brawler': 2, 'Cybernetic': 5, 'Set3_Blademaster': 3, 'Vanguard': 1, 'Blaster': 1, 'ManaReaver': 1, 'Starship': 1}"
45966,KR_4345979831,"{'Malphite': {'items': [], 'star': 2}, 'KhaZix': {'items': [], 'star': 2}, 'Graves': {'items': [], 'star': 2}, 'KaiSa': {'items': [], 'star': 2}, 'Shaco': {'items': [], 'star': 2}}",{},"{'Rebel': 1, 'Set3_Brawler': 1, 'Set3_Void': 1, 'Infiltrator': 3, 'SpacePirate': 1, 'Blaster': 1, 'Valkyrie': 1, 'DarkStar': 1}"
71470,KR_4294692860,"{'TwistedFate': {'items': [], 'star': 2}, 'Malphite': {'items': [], 'star': 2}, 'KhaZix': {'items': [], 'star': 2}, 'Blitzcrank': {'items': [], 'star': 2}, 'Vi': {'items': [], 'star': 1}, 'ChoGath': {'items': [], 'star': 2}, 'VelKoz': {'items': [], 'star': 2}}",{},"{'Chrono': 2, 'Set3_Sorcerer': 2, 'Rebel': 1, 'Set3_Brawler': 4, 'Set3_Void': 3, 'Infiltrator': 1, 'Cybernetic': 1}"
85620,KR_4400073539,"{'KhaZix': {'items': [], 'star': 2}, 'KaiSa': {'items': [], 'star': 2}, 'Annie': {'items': [], 'star': 2}, 'Shaco': {'items': [], 'star': 2}, 'Rumble': {'items': [], 'star': 1}, 'WuKong': {'items': [], 'star': 2}}",{},"{'Set3_Void': 1, 'Infiltrator': 3, 'Valkyrie': 1, 'MechPilot': 2, 'Set3_Sorcerer': 1, 'DarkStar': 1, 'Demolitionist': 1, 'Chrono': 1, 'Vanguard': 1}"


In [142]:
df_items_info.head(55)

,id,name
0,1,B.F. Sword
1,2,Recurve Bow
2,3,Needlessly Large Rod
3,4,Tear of the Goddess
4,5,Chain Vest
5,6,Negatron Cloak
6,7,Giant's Belt
7,8,Spatula
8,9,Sparring Gloves
9,11,Deathblade


In [143]:
len(df_items_info)

54

In [144]:
df_items_info['name']

0                    B.F. Sword
1                   Recurve Bow
2          Needlessly Large Rod
3           Tear of the Goddess
4                    Chain Vest
5                Negatron Cloak
6                  Giant's Belt
7                       Spatula
8               Sparring Gloves
9                    Deathblade
10                 Giant Slayer
11             Hextech Gunblade
12              Spear of Shojin
13               Guardian Angel
14                Bloodthirster
15                Zeke's Herald
16     Blade of the Ruined King
17                Infinity Edge
18             Rapid Firecannon
19          Guinsoo's Rageblade
20                 Statikk Shiv
21              Titan's Resolve
22           Runaan's Hurricane
23                Zz'Rot Portal
24         Infiltrator's Talons
25                 Last Whisper
26           Rabadon's Deathcap
27                 Luden's Echo
28    Locket of the Iron Solari
29                  Ionic Spark
30               Morellonomicon
31      

In [145]:
# champion 컬럼의 챔피언 + 아이템 정보를 이용해
# 기본 시너지와 아이템 추가 시너지를 함께 세어 combination 형태로 만드는 함수v2

def make_combination_from_champion_and_items(champion_dict):
    # 시너지 카운트 저장용 딕셔너리
    trait_count = {}
    
    # 챔피언 이름, 정보 하나씩 반복
    for champ_name, champ_info in champion_dict.items():
        # 챔피언 기본 시너지 조회
        base_traits = champion_traits.get(champ_name, [])

        # 기본 시너지 개수 누적
        for trait in base_traits:
            trait_count[trait] = trait_count.get(trait, 0) + 1
        
        # 장착 아이템 목록 조회
        item_ids = champ_info.get('items', [])
        
        # 아이템이 추가 시너지를 주는지 확인
        for item_id in item_ids:
            added_trait = item_to_trait.get(item_id)

            # 추가 시너지가 있으면 개수 누적
            if added_trait:
                trait_count[added_trait] = trait_count.get(added_trait, 0) + 1
    
    # 시너지별 개수 반환
    return trait_count

In [146]:
# 혹시 모르니까 revuilt_v2 컬럼 만들기.
df_all_match_2['combination_rebuilt_v2'] = df_all_match_2['combination'].copy()

In [147]:
#채울 대상
mask = (
    (df_all_match_2['champion'] != {}) &
    (df_all_match_2['combination'] == {})
)

#아이템 반영한 함수 실행
df_all_match_2.loc[mask, 'combination_rebuilt_v2'] = (  #여기만 수정
    df_all_match_2.loc[mask, 'champion']
    .apply(make_combination_from_champion_and_items)
)

# 개수 계산
original_empty = (df_all_match_2['combination'] == {}).sum()
rebuilt_empty = (df_all_match_2['combination_rebuilt_v2'] == {}).sum()
filled_count = original_empty - rebuilt_empty
total_rows = len(df_all_match_2)

# 비율 계산
original_empty_ratio = original_empty / total_rows * 100
rebuilt_empty_ratio = rebuilt_empty / total_rows * 100
filled_ratio_total = filled_count / total_rows * 100
filled_ratio_from_empty = filled_count / original_empty * 100 if original_empty != 0 else 0

# 출력
print("=========함수v1 실행 결과=========")
print(f"원본 combination 빈값 개수: {original_empty} ({original_empty_ratio:.2f}%)")
print(f"새 컬럼 combination_rebuilt 빈값 개수: {rebuilt_empty} ({rebuilt_empty_ratio:.2f}%)")
print(f"채운 개수: {filled_count} ({filled_ratio_total:.2f}% / 빈값 기준 {filled_ratio_from_empty:.2f}%)")


=========함수v1 실행 결과=========
원본 combination 빈값 개수: 380 (0.10%)
새 컬럼 combination_rebuilt 빈값 개수: 263 (0.07%)
채운 개수: 117 (0.03% / 빈값 기준 30.79%)


In [148]:
#확인
df_all_match_2.loc[
    mask,
    ['gameid', 'champion', 'combination', 'combination_rebuilt_v2']
].head(10)

,gameid,champion,combination,combination_rebuilt_v2
6627,KR_4271274609,"{'Fiora': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 2}, 'KaiSa': {'items': [], 'star': 1}, 'Mordekaiser': {'items': [], 'star': 2}, 'Kassadin': {'items': [], 'star': 1}}",{},"{'Cybernetic': 2, 'Set3_Blademaster': 1, 'Vanguard': 2, 'Valkyrie': 1, 'Infiltrator': 1, 'DarkStar': 1, 'Set3_Celestial': 1, 'ManaReaver': 1}"
10005,KR_4359182660,"{'JarvanIV': {'items': [], 'star': 1}, 'Lux': {'items': [], 'star': 1}}",{},"{'DarkStar': 2, 'Protector': 1, 'Set3_Sorcerer': 1}"
15897,KR_4320179268,"{'Malphite': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 1}, 'Mordekaiser': {'items': [], 'star': 1}, 'Ashe': {'items': [], 'star': 1}}",{},"{'Rebel': 1, 'Set3_Brawler': 1, 'Cybernetic': 1, 'Vanguard': 2, 'DarkStar': 1, 'Set3_Celestial': 1, 'Sniper': 1}"
20909,KR_4380989360,"{'Fiora': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [], 'star': 2}, 'Irelia': {'items': [16, 19, 29], 'star': 2}, 'WuKong': {'items': [], 'star': 2}, 'Thresh': {'items': [], 'star': 1}, 'Ekko': {'items': [], 'star': 2}}",{},"{'Cybernetic': 6, 'Set3_Blademaster': 2, 'Vanguard': 2, 'Blaster': 1, 'Set3_Brawler': 1, 'ManaReaver': 2, 'Chrono': 2, 'Infiltrator': 1}"
22014,KR_4366152411,"{'Malphite': {'items': [], 'star': 2}, 'KhaZix': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 3}, 'Mordekaiser': {'items': [], 'star': 2}, 'Shaco': {'items': [], 'star': 2}, 'Jayce': {'items': [], 'star': 1}, 'WuKong': {'items': [], 'star': 2}}",{},"{'Rebel': 1, 'Set3_Brawler': 1, 'Set3_Void': 1, 'Infiltrator': 2, 'Cybernetic': 1, 'Vanguard': 4, 'DarkStar': 2, 'SpacePirate': 1, 'Chrono': 1}"
41248,KR_4367362223,"{'Darius': {'items': [], 'star': 2}, 'Rakan': {'items': [], 'star': 2}, 'XinZhao': {'items': [], 'star': 2}, 'Jayce': {'items': [], 'star': 3}, 'Kassadin': {'items': [], 'star': 2}, 'Ashe': {'items': [], 'star': 2}, 'Jhin': {'items': [], 'star': 1}}",{},"{'SpacePirate': 2, 'ManaReaver': 2, 'Set3_Celestial': 4, 'Protector': 2, 'Vanguard': 1, 'Sniper': 2, 'DarkStar': 1}"
42176,KR_4289735897,"{'Malphite': {'items': [], 'star': 2}, 'Fiora': {'items': [], 'star': 2}, 'Leona': {'items': [], 'star': 2}, 'Yasuo': {'items': [], 'star': 2}, 'Lucian': {'items': [], 'star': 2}, 'Vi': {'items': [], 'star': 2}, 'Irelia': {'items': [], 'star': 1}, 'AurelionSol': {'items': [], 'star': 2}}",{},"{'Rebel': 3, 'Set3_Brawler': 2, 'Cybernetic': 5, 'Set3_Blademaster': 3, 'Vanguard': 1, 'Blaster': 1, 'ManaReaver': 1, 'Starship': 1}"
45966,KR_4345979831,"{'Malphite': {'items': [], 'star': 2}, 'KhaZix': {'items': [], 'star': 2}, 'Graves': {'items': [], 'star': 2}, 'KaiSa': {'items': [], 'star': 2}, 'Shaco': {'items': [], 'star': 2}}",{},"{'Rebel': 1, 'Set3_Brawler': 1, 'Set3_Void': 1, 'Infiltrator': 3, 'SpacePirate': 1, 'Blaster': 1, 'Valkyrie': 1, 'DarkStar': 1}"
71470,KR_4294692860,"{'TwistedFate': {'items': [], 'star': 2}, 'Malphite': {'items': [], 'star': 2}, 'KhaZix': {'items': [], 'star': 2}, 'Blitzcrank': {'items': [], 'star': 2}, 'Vi': {'items': [], 'star': 1}, 'ChoGath': {'items': [], 'star': 2}, 'VelKoz': {'items': [], 'star': 2}}",{},"{'Chrono': 2, 'Set3_Sorcerer': 2, 'Rebel': 1, 'Set3_Brawler': 4, 'Set3_Void': 3, 'Infiltrator': 1, 'Cybernetic': 1}"
85620,KR_4400073539,"{'KhaZix': {'items': [], 'star': 2}, 'KaiSa': {'items': [], 'star': 2}, 'Annie': {'items': [], 'star': 2}, 'Shaco': {'items': [], 'star': 2}, 'Rumble': {'items': [], 'star': 1}, 'WuKong': {'items': [], 'star': 2}}",{},"{'Set3_Void': 1, 'Infiltrator': 3, 'Valkyrie': 1, 'MechPilot': 2, 'Set3_Sorcerer': 1, 'DarkStar': 1, 'Demolitionist': 1, 'Chrono': 1, 'Vanguard': 1}"


In [149]:
mask_eval = df_all_match_2['combination'] != {}

df_all_match_2.loc[mask_eval, 'combination_rebuilt_v2_test'] = (
    df_all_match_2.loc[mask_eval, 'champion']
    .apply(make_combination_from_champion_and_items)
)

exact_match_ratio = (
    df_all_match_2.loc[mask_eval, 'combination'] ==
    df_all_match_2.loc[mask_eval, 'combination_rebuilt_v2_test']
).mean()

print(exact_match_ratio)

0.98798516125776


#여기까지

In [150]:
import ast
import json
import pandas as pd

# 1. 시너지별 활성화 기준(Threshold) 정의
synergy_thresholds = {
    # 직업 시너지
    'Set3_Blademaster': [3, 6, 9],
    'ManaReaver': [2],
    'Set3_Sorcerer': [2, 4, 6, 8],
    'Vanguard': [2, 4],
    'Protector': [2, 4, 6],
    'Set3_Mystic': [2, 4],
    'Set3_Brawler': [2, 4],
    'Mercenary': [1],
    'Starship': [1],
    'Infiltrator': [2, 4, 6],
    'Sniper': [2],
    'Blaster': [2, 4],
    'Demolitionist': [2],
    
    # 계열 시너지
    'Set3_Void': [3],
    'MechPilot': [3],
    'Rebel': [3, 6, 9],
    'Valkyrie': [2],
    'StarGuardian': [3, 6],
    'Cybernetic': [3, 6],
    'Chrono': [2, 4, 6],
    'DarkStar': [3, 6, 9],
    'SpacePirate': [2, 4],
    'Set3_Celestial': [2, 4, 6]
}

# 2. 활성화 레벨 계산 함수
def get_active_level(count, thresholds):
    active = 0
    for t in thresholds:
        if count >= t:
            active = t
        else:
            break
    return active

# 3. 딕셔너리(JSON) 형태로 활성화된 시너지만 반환하는 함수
def parse_active_synergies_to_dict(comb_str):
    if pd.isna(comb_str) or comb_str == '{}':
        return '{}'
        
    try:
        comb_dict = ast.literal_eval(comb_str)
    except:
        return '{}'
        
    active_res = {}
    for syn, count in comb_dict.items():
        if syn in synergy_thresholds:
            active_lvl = get_active_level(count, synergy_thresholds[syn])
            # 활성화 레벨이 0보다 큰 경우(최소 조건 달성)에만 딕셔너리에 추가
            if active_lvl > 0:
                active_res[syn] = active_lvl
                
    # 활성화된 시너지가 하나도 없으면 빈 딕셔너리 문자열 반환
    if not active_res:
        return '{}'
        
    # 기존 combination 컬럼과 동일하게 JSON 문자열 형태로 반환
    return json.dumps(active_res, ensure_ascii=False)

# 4. 파생 변수 컬럼 'active_synergy' 생성
df_all_match_2['active_synergy'] = df_all_match_2['combination'].apply(parse_active_synergies_to_dict)

# ✅ 결과 확인 (기존 combination과 변환된 active_synergy 비교)
display(df_all_match_2[['combination', 'active_synergy']].head(10))

,combination,active_synergy
0,"{'Cybernetic': 1, 'Demolitionist': 1, 'Infiltrator': 1, 'Rebel': 1, 'Set3_Brawler': 1, 'Set3_Celestial': 1, 'Set3_Void': 1, 'Sniper': 1}",{}
1,"{'Blaster': 1, 'Chrono': 1, 'Cybernetic': 4, 'Demolitionist': 1, 'Rebel': 1, 'Set3_Blademaster': 2, 'Set3_Brawler': 1, 'Set3_Sorcerer': 1, 'Set3_Void': 1, 'Valkyrie': 1, 'Vanguard': 2}",{}
2,"{'Blaster': 1, 'Cybernetic': 1, 'DarkStar': 2, 'Infiltrator': 1, 'Mercenary': 1, 'Set3_Blademaster': 1, 'Set3_Mystic': 1, 'Valkyrie': 1}",{}
3,"{'DarkStar': 1, 'Protector': 2, 'Set3_Blademaster': 1, 'Set3_Celestial': 5, 'Set3_Mystic': 1, 'Sniper': 1, 'StarGuardian': 2, 'Vanguard': 2}",{}
4,"{'Blaster': 1, 'Chrono': 5, 'DarkStar': 3, 'Protector': 1, 'Set3_Blademaster': 1, 'Set3_Brawler': 1, 'Set3_Sorcerer': 2, 'Sniper': 2}",{}
5,"{'Rebel': 1, 'Set3_Blademaster': 1, 'Set3_Celestial': 1, 'Sniper': 1, 'StarGuardian': 1, 'Vanguard': 1}",{}
6,"{'Protector': 1, 'Set3_Mystic': 1, 'Set3_Sorcerer': 3, 'StarGuardian': 6, 'Vanguard': 1}",{}
7,"{'Blaster': 1, 'Cybernetic': 1, 'Infiltrator': 1, 'Set3_Blademaster': 1, 'Set3_Celestial': 1, 'Set3_Sorcerer': 1, 'StarGuardian': 1, 'Valkyrie': 1}",{}
8,"{'Blaster': 1, 'DarkStar': 2, 'Set3_Celestial': 1, 'Set3_Sorcerer': 1, 'Set3_Void': 1, 'Sniper': 2, 'SpacePirate': 1, 'StarGuardian': 1, 'Vanguard': 2}",{}
9,"{'Chrono': 4, 'DarkStar': 3, 'Infiltrator': 1, 'Rebel': 1, 'Set3_Blademaster': 1, 'Set3_Brawler': 2, 'Set3_Sorcerer': 1, 'Sniper': 2, 'Vanguard': 1}",{}
